# NYC Yellow Taxi benchmarks

Set `MONTH` below to compare the official TLC Parquet file with an equivalent local CSV, measure Pandas memory optimization, and verify daily pickup counts by zone against ClickHouse. CSV/benchmark artifacts are cached under `.cache/`.

In [1]:
from collections import Counter
from pathlib import Path
from statistics import median
from time import perf_counter
import os

import clickhouse_connect
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from dotenv import load_dotenv
from IPython.display import display

from load_month import download_source, parse_month

ROOT = Path.cwd()
load_dotenv(ROOT / ".env")
MONTH = os.getenv("NYC_TAXI_MONTH", "2026-09")
source_month = parse_month(MONTH)
month_start = pd.Timestamp(source_month)
month_end = month_start + pd.offsets.MonthBegin(1)
start_text, end_text = month_start.strftime("%Y-%m-%d"), month_end.strftime("%Y-%m-%d")
TIMEZONE = "America/New_York"
cache_dir = ROOT / ".cache" / "tlc"
source_path = download_source(source_month, cache_dir)
parquet_file = pq.ParquetFile(source_path)
client = clickhouse_connect.get_client(
    host=os.getenv("CLICKHOUSE_HOST", "127.0.0.1"),
    port=int(os.getenv("CLICKHOUSE_PORT", "8123")),
    database=os.environ["CLICKHOUSE_DATABASE"],
    username=os.environ["CLICKHOUSE_USER"],
    password=os.environ["CLICKHOUSE_PASSWORD"],
)
print(f"Source month: {MONTH}; Parquet metadata rows: {parquet_file.metadata.num_rows:,}")

Source month: 2026-09; Parquet metadata rows: 3,876,044


## Equivalent CSV and format-read timings

Both readers use the same selected columns and every row in the month. CSV is generated incrementally from Parquet batches. Reads use explicit CSV dtypes and timestamp parsing rather than type inference. Repeated local reads share the operating system file cache; no cache-clearing is attempted.

In [2]:
benchmark_columns = [
    "VendorID", "tpep_pickup_datetime", "passenger_count", "trip_distance",
    "RatecodeID", "store_and_fwd_flag", "PULocationID", "DOLocationID",
    "payment_type", "fare_amount", "tip_amount", "total_amount",
]
missing_columns = set(benchmark_columns) - set(parquet_file.schema_arrow.names)
if missing_columns:
    raise ValueError(f"Source month is missing benchmark columns: {sorted(missing_columns)}")
csv_path = ROOT / ".cache" / "benchmarks" / f"yellow_tripdata_{MONTH}_benchmark.csv"
csv_path.parent.mkdir(parents=True, exist_ok=True)
if csv_path.exists():
    csv_path.unlink()
csv_rows = 0
write_header = True
for batch in parquet_file.iter_batches(batch_size=65_536, columns=benchmark_columns):
    batch_frame = batch.to_pandas()
    batch_frame.to_csv(csv_path, mode="w" if write_header else "a", header=write_header, index=False)
    csv_rows += len(batch_frame)
    write_header = False
assert csv_rows == parquet_file.metadata.num_rows
print(f"Equivalent CSV rows: {csv_rows:,}; columns: {len(benchmark_columns)}")

Equivalent CSV rows: 3,876,044; columns: 12


In [3]:
csv_dtypes = {
    "VendorID": "Int64", "passenger_count": "Int64", "RatecodeID": "Int64",
    "store_and_fwd_flag": "string", "PULocationID": "Int64", "DOLocationID": "Int64",
    "payment_type": "Int64", "trip_distance": "float64", "fare_amount": "float64",
    "tip_amount": "float64", "total_amount": "float64",
}

def count_parquet_rows():
    return sum(batch.num_rows for batch in parquet_file.iter_batches(
        batch_size=65_536, columns=benchmark_columns
    ))

def count_csv_rows():
    return sum(len(chunk) for chunk in pd.read_csv(
        csv_path, usecols=benchmark_columns, dtype=csv_dtypes,
        parse_dates=["tpep_pickup_datetime"], chunksize=65_536,
    ))

parquet_times, csv_times = [], []
for _ in range(3):
    started = perf_counter()
    parquet_count = count_parquet_rows()
    parquet_times.append(perf_counter() - started)
    started = perf_counter()
    csv_count = count_csv_rows()
    csv_times.append(perf_counter() - started)
    assert parquet_count == csv_count == parquet_file.metadata.num_rows

format_results = pd.DataFrame([
    {"format": "Parquet / PyArrow batches", "size_MB": source_path.stat().st_size / 1e6,
     "median_read_seconds": median(parquet_times), "rows": parquet_count,
     "compression": ", ".join(sorted(set(parquet_file.metadata.row_group(0).column(i).compression
         for i in range(parquet_file.metadata.row_group(0).num_columns))))},
    {"format": "CSV / Pandas chunks", "size_MB": csv_path.stat().st_size / 1e6,
     "median_read_seconds": median(csv_times), "rows": csv_count, "compression": "none"},
])
display(format_results)

,format,size_MB,median_read_seconds,rows,compression
0,Parquet / PyArrow batches,67.374896,0.066702,3876044,ZSTD
1,CSV / Pandas chunks,220.499852,6.639290,3876044,none


## Chunked CSV and Pandas memory

The chunked read verifies the full row count without retaining the whole CSV. Memory optimization uses the first 250,000 rows. IDs are narrowed only after range checks; low-cardinality codes become categories. Monetary columns remain `float64` and are checked for exact equality.

In [4]:
chunk_size = 65_536
chunked_rows = sum(len(chunk) for chunk in pd.read_csv(
    csv_path, usecols=benchmark_columns, dtype=csv_dtypes,
    parse_dates=["tpep_pickup_datetime"], chunksize=chunk_size,
))
assert chunked_rows == parquet_file.metadata.num_rows
print(f"Chunk size: {chunk_size:,}; chunked rows processed: {chunked_rows:,}")

sample = pd.read_csv(
    csv_path, usecols=benchmark_columns, dtype=csv_dtypes,
    parse_dates=["tpep_pickup_datetime"], nrows=250_000,
)
money_columns = ["fare_amount", "tip_amount", "total_amount"]
money_before = sample[money_columns].copy(deep=True)
memory_before = sample.memory_usage(deep=True).sum()

range_checks = []
for column, target_dtype in [("passenger_count", "Int8"), ("PULocationID", "Int16"), ("DOLocationID", "Int16")]:
    values = sample[column]
    minimum, maximum = values.min(), values.max()
    if pd.notna(minimum) and pd.notna(maximum):
        limits = np.iinfo(np.dtype(target_dtype.lower()))
        if minimum < limits.min or maximum > limits.max:
            raise ValueError(f"{column} values do not fit {target_dtype}: {minimum}..{maximum}")
    original = values.astype("Int64")
    sample[column] = values.astype(target_dtype)
    assert sample[column].astype("Int64").equals(original)
    range_checks.append({"column": column, "dtype": target_dtype, "min": minimum, "max": maximum,
                         "missing": int(values.isna().sum())})

category_columns = ["VendorID", "RatecodeID", "payment_type", "store_and_fwd_flag"]
for column in category_columns:
    sample[column] = sample[column].astype("category")
memory_after = sample.memory_usage(deep=True).sum()
assert sample[money_columns].equals(money_before)
memory_results = pd.DataFrame([
    {"state": "before", "memory_MB": memory_before / 1e6},
    {"state": "after", "memory_MB": memory_after / 1e6},
])
memory_saved_pct = 100 * (memory_before - memory_after) / memory_before
display(pd.DataFrame(range_checks))
display(memory_results)
print(f"Memory saved: {memory_saved_pct:.1f}%; categories: {category_columns}; monetary precision unchanged")

Chunk size: 65,536; chunked rows processed: 3,876,044


,column,dtype,min,max,missing
0,passenger_count,Int8,0,6,0
1,PULocationID,Int16,1,265,0
2,DOLocationID,Int16,1,265,0


,state,memory_MB
0,before,38.000132
1,after,13.000925


Memory saved: 65.8%; categories: ['VendorID', 'RatecodeID', 'payment_type', 'store_and_fwd_flag']; monetary precision unchanged


## ClickHouse versus Pandas daily pickup counts

Both paths use the selected source month and NYC-local half-open date interval. NULL pickup-zone IDs map to `0`; rows with NULL pickup timestamps are excluded. ClickHouse wall time includes server execution and result transfer/decoding. Pandas data-read time measures PyArrow batch iteration; aggregation time includes conversion, filtering, and grouping.

In [5]:
comparison_sql = f"""
SELECT toDate(pickup_datetime, '{TIMEZONE}') AS pickup_date,
       ifNull(pickup_location_id, toInt32(0)) AS pickup_location_id,
       count() AS trip_count
FROM trips
WHERE source_month = toDate('{source_month:%Y-%m-%d}')
  AND isNotNull(pickup_datetime)
  AND pickup_datetime >= toDateTime64('{start_text} 00:00:00', 6, '{TIMEZONE}')
  AND pickup_datetime < toDateTime64('{end_text} 00:00:00', 6, '{TIMEZONE}')
GROUP BY pickup_date, pickup_location_id
ORDER BY pickup_date, pickup_location_id
"""

def pandas_daily_counts():
    counts = Counter()
    data_read_seconds = aggregation_seconds = 0.0
    batches = iter(parquet_file.iter_batches(
        batch_size=65_536, columns=["tpep_pickup_datetime", "PULocationID"]
    ))
    while True:
        started = perf_counter()
        try:
            batch = next(batches)
        except StopIteration:
            data_read_seconds += perf_counter() - started
            break
        data_read_seconds += perf_counter() - started
        started = perf_counter()
        frame = batch.to_pandas()
        pickups = pd.to_datetime(frame["tpep_pickup_datetime"])
        valid = pickups.notna() & (pickups >= month_start) & (pickups < month_end)
        subset = pd.DataFrame({
            "pickup_date": pickups.loc[valid].dt.date,
            "pickup_location_id": frame.loc[valid, "PULocationID"].fillna(0).astype("int64"),
        })
        grouped = subset.groupby(["pickup_date", "pickup_location_id"], sort=True).size()
        counts.update({key: int(value) for key, value in grouped.items()})
        aggregation_seconds += perf_counter() - started
    return counts, data_read_seconds, aggregation_seconds

clickhouse_times, pandas_times, pandas_read_times, pandas_aggregation_times = [], [], [], []
for _ in range(3):
    started = perf_counter()
    clickhouse_rows = client.query(comparison_sql).result_rows
    clickhouse_times.append(perf_counter() - started)
    pandas_counts, pandas_read, pandas_aggregation = pandas_daily_counts()
    pandas_read_times.append(pandas_read)
    pandas_aggregation_times.append(pandas_aggregation)
    pandas_times.append(pandas_read + pandas_aggregation)

clickhouse_result = tuple((row[0], row[1], int(row[2])) for row in clickhouse_rows)
pandas_result = tuple(sorted((day, zone, count) for (day, zone), count in pandas_counts.items()))
assert clickhouse_result == pandas_result, "ClickHouse and Pandas results differ"
assert sum(row[2] for row in clickhouse_result) <= parquet_file.metadata.num_rows
timing_results = pd.DataFrame([
    {"path": "ClickHouse SQL + result transfer", "median_seconds": median(clickhouse_times)},
    {"path": "Pandas Parquet read", "median_seconds": median(pandas_read_times)},
    {"path": "Pandas aggregation", "median_seconds": median(pandas_aggregation_times)},
    {"path": "Pandas end-to-end", "median_seconds": median(pandas_times)},
])
display(timing_results)
print(f"Groups on each path: {len(clickhouse_result):,}; exact equality: PASS; mismatches: 0")
client.close()

,path,median_seconds
0,ClickHouse SQL + result transfer,0.075479
1,Pandas Parquet read,0.052089
2,Pandas aggregation,0.979441
3,Pandas end-to-end,1.030450


Groups on each path: 7,270; exact equality: PASS; mismatches: 0


## Recreate the visualizations

The standalone Plotly charts are generated from ClickHouse by running `.venv/bin/python plot_charts.py --month 2026-09`. Their assignments and metric definitions are documented in the project README.